# Evaluation Pipeline — Reference-Based Comparison

Compare synthesised audio (DDSP and baseline) against a **solo violin reference set**
using pre-computed feature parquets from the *feature_extraction* notebook.

1. **Load features** — read reference and synthesised feature parquets.
2. **Strategy 1 (Distribution)** — compare concatenated folder-level distributions to the reference.
3. **Strategy 2 (Pairwise)** — compare each individual file to the reference distribution.
4. **Visualizations** — bar charts, box plots, method comparison.

In [3]:
import logging
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")

print(f"Project root: {PROJECT_ROOT}")

Project root: /m/home/home3/37/thieun1/unix/Project/final_project


In [4]:
from evaluation.batch_inference import load_features, evaluate_dir
from evaluation.loss import Loss
from visualize import (
    plot_distribution_comparison,
    plot_loss_by_group,
    plot_loss_boxplot,
)

import numpy as np
import pandas as pd

2026-04-18 11:44:46.163463: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-18 11:44:46.250229: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-18 11:44:56.766636: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /u/37/thieun1/unix/anaconda3/envs/conda_env3.10/lib
2026-04-18 11:44:56.766880: W tensorflow/compi

## Configuration

Point each path to a `.parquet` file produced by the *feature_extraction* notebook.
Set `FEATURE_COLS` to choose which columns are compared (pitch columns like
`f0_hz` / `f0_confidence` are excluded by default).

In [ ]:
FEATURES_DIR = PROJECT_ROOT / "data" / "processed" / "normalized"

# --- Pre-computed feature parquets (from feature_extraction notebook) ---
REF_PARQUET      = FEATURES_DIR / "bach_features_norm.parquet"
DDSP_PARQUET     = FEATURES_DIR / "transfered_features_norm.parquet"
BASELINE_PARQUET = FEATURES_DIR / "baseline_features_norm.parquet"

# --- Feature columns to compare (set to None to use all numeric columns) ---
FEATURE_COLS = [
    "spectral_centroid",
    "spectral_crest",
    "spectral_decrease",
    "spectral_flatness",
    "spectral_roll_off",
    "spectral_skewness",
    "spectral_spread",
]

# --- Result CSVs ---
RESULTS_DIR = PROJECT_ROOT / "artifacts" / "evaluation"

for name, path in [("Bach", REF_PARQUET), ("DDSP", DDSP_PARQUET), ("Baseline", BASELINE_PARQUET)]:
    print(f"{name:10s}: {path}  (exists: {path.exists()})")
print(f"Features  : {FEATURE_COLS}")
print(f"Results   : {RESULTS_DIR}")

Bach      : /m/home/home3/37/thieun1/unix/Project/final_project/data/processed/bach_features.parquet  (exists: True)
DDSP      : /m/home/home3/37/thieun1/unix/Project/final_project/data/processed/transfered_features.parquet  (exists: True)
Baseline  : /m/home/home3/37/thieun1/unix/Project/final_project/data/processed/baseline_features.parquet  (exists: True)
Features  : ['spectral_centroid', 'spectral_crest', 'spectral_decrease', 'spectral_flatness', 'spectral_roll_off', 'spectral_skewness', 'spectral_spread']
Results   : /m/home/home3/37/thieun1/unix/Project/final_project/artifacts/evaluation


## 1. Load Pre-Computed Features

In [26]:
ref_df = load_features(REF_PARQUET)
ddsp_df = load_features(DDSP_PARQUET)
baseline_df = load_features(BASELINE_PARQUET)

# Build feature matrices using only the selected columns
feat_keys = FEATURE_COLS
ref_2d = ref_df[feat_keys].to_numpy(dtype=np.float64)

print(f"Reference : {ref_2d.shape[0]:,} frames, {ref_2d.shape[1]} features, {ref_df['filename'].nunique()} files")
print(f"DDSP      : {len(ddsp_df):,} frames, {ddsp_df['filename'].nunique()} files")
print(f"Baseline  : {len(baseline_df):,} frames, {baseline_df['filename'].nunique()} files")
print(f"Feature keys: {feat_keys}")

Reference : 50,347 frames, 7 features, 21 files
DDSP      : 106,805 frames, 3094 files
Baseline  : 106,856 frames, 3094 files
Feature keys: ['spectral_centroid', 'spectral_crest', 'spectral_decrease', 'spectral_flatness', 'spectral_roll_off', 'spectral_skewness', 'spectral_spread']


## 2. Strategy 1 — Distribution-Level Comparison

Compare the concatenated folder-level feature distribution of each method
against the reference.

In [ ]:
SYNTH_DATA = {"ddsp": ddsp_df, "baseline": baseline_df}

loss = Loss()
dist_rows = []

for method_name, synth_df in SYNTH_DATA.items():
    synth_2d = synth_df[feat_keys].to_numpy(dtype=np.float64)
    dist_result = loss.evaluate(synth_2d, ref_2d)

    dist_rows.append({
        "method": method_name,
        "n_files": synth_df["filename"].nunique(),
        "total_frames": synth_2d.shape[0],
        **dist_result,
    })
    print(f"{method_name}: mmd={dist_result['mmd']:.6f}, wasserstein={dist_result['wasserstein']:.6f}")

df_distribution = pd.DataFrame(dist_rows)
display(df_distribution)

plot_distribution_comparison(df_distribution, "mmd")
plot_distribution_comparison(df_distribution, "wasserstein")

### Strategy 1b — Per-Feature Distribution

Run the same distribution-level comparison **one feature at a time** (1-D MMD &
Wasserstein) so we can see which individual features differ most from the reference.

In [28]:
loss_1d = Loss()
per_feat_dist_rows = []

for method_name, synth_df in SYNTH_DATA.items():
    for feat in feat_keys:
        synth_1d = synth_df[feat].to_numpy(dtype=np.float64)
        ref_1d = ref_df[feat].to_numpy(dtype=np.float64)
        result = loss_1d.evaluate(synth_1d, ref_1d)
        per_feat_dist_rows.append({
            "method": method_name,
            "feature": feat,
            "n_files": synth_df["filename"].nunique(),
            "total_frames": synth_1d.shape[0],
            **result,
        })

df_distribution_per_feature = pd.DataFrame(per_feat_dist_rows)
display(df_distribution_per_feature.pivot(index="feature", columns="method", values="mmd").round(4))
display(df_distribution_per_feature.pivot(index="feature", columns="method", values="wasserstein").round(4))

method,baseline,ddsp
feature,,
spectral_centroid,0.4916,0.4800
spectral_crest,0.7114,0.6584
spectral_decrease,0.2687,0.4090
spectral_flatness,0.9182,0.9277
spectral_roll_off,0.4573,0.5830
spectral_skewness,0.5057,0.3330
spectral_spread,0.4927,0.6122


method,baseline,ddsp
feature,,
spectral_centroid,492.9004,534.6135
spectral_crest,50.6740,47.6943
spectral_decrease,0.0064,0.0077
spectral_flatness,0.2048,0.2816
spectral_roll_off,839.1756,1394.4940
spectral_skewness,0.5616,0.4202
spectral_spread,209.4162,390.9140


## 3. Strategy 2 — Pairwise File-vs-Reference

Compare each individual synthesised file's feature distribution against the
full reference distribution.

In [ ]:
from joblib import Parallel, delayed

N_REF_SUBSAMPLE = 10_000

pair_loss = Loss()
rng = np.random.default_rng(0)
idx = rng.choice(ref_2d.shape[0], size=min(N_REF_SUBSAMPLE, ref_2d.shape[0]), replace=False)
ref_2d_sub = ref_2d[idx]
print(f"Pairwise reference: {ref_2d_sub.shape[0]:,} frames (subsampled from {ref_2d.shape[0]:,})")

def _eval_file(method_name, fname, file_2d, ref, loss_obj):
    if file_2d.shape[0] < 2:
        return None
    distances = loss_obj.evaluate(file_2d, ref)
    return {"method": method_name, "file": fname, **distances}

tasks = [
    (method_name, fname, group[feat_keys].to_numpy(dtype=np.float64))
    for method_name, synth_df in SYNTH_DATA.items()
    for fname, group in synth_df.groupby("filename")
]

results = Parallel(n_jobs=-1, backend="loky", verbose=10)(
    delayed(_eval_file)(m, f, x, ref_2d_sub, pair_loss) for m, f, x in tasks
)

pair_rows = [r for r in results if r is not None]
df_pairwise = pd.DataFrame(pair_rows)
print(f"Pairwise results: {len(df_pairwise)} rows")
df_pairwise.head()

### Pairwise Summary by Method

In [30]:
# display(df_pairwise.groupby("method")[["mmd", "wasserstein"]].describe().round(4))

# plot_loss_boxplot(df_pairwise, "method", "mmd")
# plot_loss_boxplot(df_pairwise, "method", "wasserstein")

### Strategy 2b — Per-Feature Pairwise

Same pairwise file-vs-reference comparison but evaluated **separately for each
1-D feature**, yielding one (method × file × feature) row.

In [31]:
pair_loss_1d = Loss()

ref_1d_sub = {feat: ref_2d_sub[:, i] for i, feat in enumerate(feat_keys)}

def _eval_file_per_feature(method_name, fname, file_2d, ref_cols, feats, loss_obj):
    if file_2d.shape[0] < 2:
        return []
    out = []
    for i, feat in enumerate(feats):
        d = loss_obj.evaluate(file_2d[:, i], ref_cols[feat])
        out.append({"method": method_name, "file": fname, "feature": feat, **d})
    return out

per_feat_tasks = [
    (method_name, fname, group[feat_keys].to_numpy(dtype=np.float64))
    for method_name, synth_df in SYNTH_DATA.items()
    for fname, group in synth_df.groupby("filename")
]

per_feat_results = Parallel(n_jobs=-1, backend="loky", verbose=10)(
    delayed(_eval_file_per_feature)(m, f, x, ref_1d_sub, feat_keys, pair_loss_1d)
    for m, f, x in per_feat_tasks
)

per_feat_pair_rows = [r for batch in per_feat_results for r in batch]
df_pairwise_per_feature = pd.DataFrame(per_feat_pair_rows)
print(f"Per-feature pairwise results: {len(df_pairwise_per_feature)} rows")
df_pairwise_per_feature.head()

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done   9 tasks      | elapsed:   20.8s
[Parallel(n_jobs=-1)]: Done  18 tasks      | elapsed:   37.5s
[Parallel(n_jobs=-1)]: Done  29 tasks      | elapsed:   38.3s
[Parallel(n_jobs=-1)]: Done  40 tasks      | elapsed:   55.3s
[Parallel(n_jobs=-1)]: Done  53 tasks      | elapsed:  1.2min
[Parallel(n_jobs=-1)]: Done  66 tasks      | elapsed:  1.5min
[Parallel(n_jobs=-1)]: Done  81 tasks      | elapsed:  1.8min
[Parallel(n_jobs=-1)]: Done  96 tasks      | elapsed:  1.8min
[Parallel(n_jobs=-1)]: Done 113 tasks      | elapsed:  2.3min
[Parallel(n_jobs=-1)]: Done 130 tasks      | elapsed:  2.6min
[Parallel(n_jobs=-1)]: Done 149 tasks      | elapsed:  2.9min
[Parallel(n_jobs=-1)]: Done 168 tasks      | elapsed:  3.2min
[Parallel(n_jobs=-1)]: Done 189 tasks      | elapsed:  3.5min
[Parallel(n_jobs=-1)]: Done 210 tasks      | elapsed:  4.0min
[Parallel(n_jobs=-1)]: Done 233 tasks      | elapsed:  

Per-feature pairwise results: 43316 rows


[Parallel(n_jobs=-1)]: Done 6188 out of 6188 | elapsed: 110.9min finished


,method,file,feature,mmd,wasserstein
0,ddsp,arpeggios_straight_a_TRANSFERED.wav,spectral_centroid,0.413911,432.712777
1,ddsp,arpeggios_straight_a_TRANSFERED.wav,spectral_crest,0.389389,33.618737
2,ddsp,arpeggios_straight_a_TRANSFERED.wav,spectral_decrease,0.519947,0.012172
3,ddsp,arpeggios_straight_a_TRANSFERED.wav,spectral_flatness,0.906666,0.283292
4,ddsp,arpeggios_straight_a_TRANSFERED.wav,spectral_roll_off,0.608837,1140.847055


In [32]:
per_feat_summary = (
    df_pairwise_per_feature
    .groupby(["method", "feature"])[["mmd", "wasserstein"]]
    .agg(["mean", "std", "median"])
    .round(4)
)
display(per_feat_summary)

mmd                 wasserstein            \
                              mean     std  median        mean       std   
method   feature                                                           
baseline spectral_centroid  0.4966  0.1530  0.5002    482.8083  177.7160   
         spectral_crest     0.7377  0.1550  0.7692     51.5169    8.8837   
         spectral_decrease  0.2837  0.1642  0.2797      0.0064    0.0010   
         spectral_flatness  0.8971  0.0912  0.9064      0.2087    0.0381   
         spectral_roll_off  0.4658  0.1519  0.4570    831.3961  288.5221   
         spectral_skewness  0.4944  0.1528  0.5055      0.5444    0.1343   
         spectral_spread    0.4875  0.1523  0.4989    207.1344   63.0704   
ddsp     spectral_centroid  0.5193  0.1725  0.5216    531.6188  199.9221   
         spectral_crest     0.7031  0.1630  0.7383     49.4737    8.8347   
         spectral_decrease  0.4102  0.1405  0.4170      0.0088    0.0038   
         spectral_flatness  0.9572  0.0801  0.9574      0.2893    0.0684   
         spectral_roll_off  0.6309  0.1813  0.6297   1398.1382  458.0021   
         spectral_skewness  0.3770  0.1910  0.3737      0.4241    0.1510   
         spectral_spread    0.6717  0.1703  0.6753    391.5117  147.8654   

                                       
                               median  
method   feature                       
baseline spectral_centroid   473.0331  
         spectral_crest       53.1007  
         spectral_decrease     0.0063  
         spectral_flatness     0.2072  
         spectral_roll_off   822.5161  
         spectral_skewness     0.5486  
         spectral_spread     206.9425  
ddsp     spectral_centroid   512.8316  
         spectral_crest       51.0628  
         spectral_decrease     0.0081  
         spectral_flatness     0.2775  
         spectral_roll_off  1386.3034  
         spectral_skewness     0.4132  
         spectral_spread     363.6361

## 4. Method Comparison

In [ ]:
plot_loss_by_group(df_pairwise, "method", "mmd")

In [ ]:
plot_loss_by_group(df_pairwise, "method", "wasserstein")

## 5. Export

Save summary tables and figures.

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR = RESULTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# CSVs
df_distribution.to_csv(RESULTS_DIR / "distribution_summary.csv", index=False) 
df_pairwise.to_csv(RESULTS_DIR / "evaluation_pairwise.csv", index=False)

pairwise_summary = df_pairwise.groupby("method")[["mmd", "wasserstein"]].agg(["mean", "std", "median"]).round(4)
pairwise_summary.to_csv(RESULTS_DIR / "pairwise_summary_by_method.csv")

# Figures
plot_distribution_comparison(df_distribution, "mmd", save_path=str(FIGURES_DIR / "distribution_mmd.png"))
plot_distribution_comparison(df_distribution, "wasserstein", save_path=str(FIGURES_DIR / "distribution_wasserstein.png"))
plot_loss_boxplot(df_pairwise, "method", "mmd", save_path=str(FIGURES_DIR / "pairwise_mmd_boxplot.png"))
plot_loss_boxplot(df_pairwise, "method", "wasserstein", save_path=str(FIGURES_DIR / "pairwise_wasserstein_boxplot.png"))
plot_loss_by_group(df_pairwise, "method", "mmd", save_path=str(FIGURES_DIR / "pairwise_mmd_by_method.png"))
plot_loss_by_group(df_pairwise, "method", "wasserstein", save_path=str(FIGURES_DIR / "pairwise_wasserstein_by_method.png"))

print(f"Exported to {RESULTS_DIR}")

In [36]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

df_distribution_per_feature.to_csv(RESULTS_DIR / "distribution_per_feature.csv", index=False)
df_pairwise_per_feature.to_csv(RESULTS_DIR / "pairwise_per_feature.csv", index=False)

per_feat_summary = (
    df_pairwise_per_feature
    .groupby(["method", "feature"])[["mmd", "wasserstein"]]
    .agg(["mean", "std", "median"])
    .round(4)
)
per_feat_summary.to_csv(RESULTS_DIR / "pairwise_per_feature_summary.csv")

print(f"Per-feature CSVs written to {RESULTS_DIR}")

Per-feature CSVs written to /m/home/home3/37/thieun1/unix/Project/final_project/artifacts/evaluation
